# HW3: Building an AI Game Master (100pts)

*Credits to Original Author: Andrew Zhu (andrz@seas.upenn.edu)*

In this homework, we'll use Kani to build a simple Game Master (GM) for the [Mausritter TTRPG](https://losing-games.itch.io/mausritter). Your GM will help walk you through creating a mouse and rolling dice to overcome challenges in the game. Along the way, you'll learn the basics of the [Kani library](https://kani.readthedocs.io/en/latest/), function calling with LLMs, and how a LLM can use structured information to act as a game state manager.

We won't implement more complex systems like combat or spellcasting in this homework, but extending this system to make a more comprehensive GM (e.g. adding inventory tracking, combat, spellcasting, structured tracking of NPCs, ...) would make for a great class final project.

## What You'll Need

- A copy of the [Mausritter rules](https://losing-games.itch.io/mausritter) (available for free on itch.io)
- Your Triton AI API Key (or [Open AI API key](https://platform.openai.com/api-keys) if you have one)
- Recommended: the [Kani documentation](https://kani.readthedocs.io/en/latest/)
- Optional: the [d20 documentation](https://d20.readthedocs.io/en/latest/start.html)

# Part 0: Setup

*Reference: [Installation (Kani Documentation)](https://kani.readthedocs.io/en/latest/install.html)*

First, install the libraries we're using and configure Kani to use the Triton AI proxy (cells below). Kani requires Python 3.10 or higher—if you encounter an error saying "No matching distribution found", you may need to upgrade your Python version.

## Google Colab (easiest)

The simplest way to run this homework is [Google Colab](https://colab.research.google.com/): upload this notebook (**File → Upload notebook**) or open it from the course repository, then run the cells in order. Colab gives you a managed Python 3.10+ environment with nothing to install on your laptop.

1. Run the **Install dependencies** cell below (`pip install …`).
2. Set your API key: in Colab, open **Secrets** (key icon in the left sidebar), add a secret named `OPENAI_API_KEY` with your Triton or OpenAI key, enable **Notebook access**, then run this once **before** the engine setup cell:

```python
import os
from google.colab import userdata
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
```

If you are not using Colab, set `OPENAI_API_KEY` on your machine as described under **Running locally** below.

## Running locally (optional)

If you prefer to run on your own computer, use a **virtual environment** so packages stay isolated.

Mac/Linux:
```shell
python3.10 -m venv ./venv
source ./venv/bin/activate
pip install jupyter
```

Windows:
```shell
python3.10 -m venv venv
venv\Scripts\activate.bat
pip install jupyter
```

You may need to restart your IDE after creating a new virtual environment for it to detect the interpreter. When the venv is active, run the same `pip install` command as in the cell below (or install `d20`, `kani[openai]`, and `openai` yourself).

### Line wrapping (VS Code)

If you use VS Code locally, you may want **Notebook Output Word Wrap** so long outputs fold to multiple lines: gear icon on the notebook → **Customize Notebook Layout** → enable **Notebook Output Word Wrap**.

In [ ]:
# Install dependencies
!pip install -q d20 'kani[openai]' openai

We'll use `api-gpt-oss-120b` as our LLM of choice (feel free to try better models if your budget allows), so we'll configure it as our *engine* (a Kani concept used to provide a standard interface to an LLM).

<div>
<img src="https://kani.readthedocs.io/en/latest/_images/concepts-figure.png" width="700"/>
</div>

If you use **Google Colab**, set `OPENAI_API_KEY` with **Secrets** as in Part 0 (or export it in a cell) so the next code cell can read `os.environ["OPENAI_API_KEY"]`.

If you run **locally**, set your API key (Triton AI or OpenAI) in the environment before starting Jupyter or VS Code—for example in the terminal:

```
export OPENAI_API_KEY=sk-helicone-cp-###########
```

With VS Code, you can launch it from the terminal and pass the key on the same line:

```
cd your/homework/dir
source venv/bin/activate
OPENAI_API_KEY=sk-helicone-cp-###########  code .
```

We'll use the `engine` global throughout the rest of this homework.

In [4]:
# Set up an engine using the Triton AI API with custom client
from kani.engines.openai import OpenAIEngine
from openai import AsyncOpenAI
import os

# Set your Triton API key as an environment variable
api_key = os.environ["OPENAI_API_KEY"]

# Create a custom OpenAI client explicitly configured for Triton API
custom_client = AsyncOpenAI(
    api_key=api_key,
    base_url="https://tritonai-api.ucsd.edu"
)

# Pass the custom client to OpenAIEngine
engine = OpenAIEngine(client=custom_client, model="api-gpt-oss-120b")


/tmp/ipykernel_43508/1078657780.py:16: UserWarning: The context length for this model was not found, defaulting to 2048 tokens. Please specify `max_context_size` if this is incorrect.
  engine = OpenAIEngine(client=custom_client, model="api-gpt-oss-120b")
/tmp/ipykernel_43508/1078657780.py:16: UserWarning: The OpenAI API type for this model was not set, defaulting to 'chat_completions' for 'api-gpt-oss-120b'. Please specify `api_type="chat_completions"` or `api_type="responses"` if this is incorrect.
  engine = OpenAIEngine(client=custom_client, model="api-gpt-oss-120b")
/home/andrewyin/UCSD-CSE-Programming-Assignments/CSE190/homeworks/hw3/venv/lib/python3.10/site-packages/kani/engines/openai/engine.py:127: UserWarning: Could not find a tokenizer for the api-gpt-oss-120b model. You may need to update tiktoken. Using o200k_base tokenizer as default.
  warnings.warn(


# Part 1: Dice Rolling & Function Calling

First, let's get familiar with the libraries we're going to use. 

## 1.1: Dice Rolling

To implement dice rolling, we'll use `d20`, a Python dice library I wrote for D&D. This library parses RPG dice notation and rolls the specified dice. For this homework, we'll use its output as Markdown, but if you're interested in a deeper dive into the library, the output is a [structured eval tree](https://d20.readthedocs.io/en/latest/expression.html).

Mausritter uses standard RPG dice notation throughout.

For example:
- `d20` means: Roll a single 20-sided die
- `1d8` means: Roll a single 8-sided die
- `3d6` means: Roll three 6-sided dice, add them together

In certain circumstances, you'll need to roll some dice and keep only the N highest (e.g. 4d6, keep highest 3). `d20` expresses this in "keep-highest" notation:

- `4d6kh3` means: Roll four six-sided dice, add the highest three together
- `2d20kl1` means: Roll two d20s, keep the lower ("disadvantage" in D&D, "advantage" in Mausritter)

For more examples, check out [the documentation](https://d20.readthedocs.io/en/latest/start.html#examples). Try running this cell a couple times to see what the result of these rolls are! Feel free to add more rolls to experiment, too.

In [5]:
from d20 import roll

print(roll("d20"))
print(roll("1d8"))
print(roll("3d6"))
print(roll("4d6kh3"))
print(roll("2d20kl1"))

1d20 (17) = `17`
1d8 (**1**) = `1`
3d6 (2, 3, 2) = `7`
4d6kh3 (**6**, ~~2~~, 3, 4) = `13`
2d20kl1 (~~**20**~~, 9) = `9`


In [ ]:
# d20 can also handle basic math:
print(roll("2d6 * 3"))
print(roll("1d6 * 10 + 1d6"))

2d6 (2, **1**) * 3 = `9`
1d6 (3) * 10 + 1d6 (**6**) = `36`


The `roll()` function returns a `RollResult` object, which contains the stringified roll (`RollResult.result`), its numeric total (`RollResult.total`), and the eval tree. You can use this, for example, to roll stats for a mouse!

Mausritter defines your mouse's stats as such:

> For each attribute, in order, roll 3d6. Keep the two highest dice results for a value between 2—12.

To make sure you understand how to use d20, let's this into a roll statement for one attribute, and print the result and final rolled total here.

In [ ]:
# TODO: roll one attribute and print its roll and value
attr_result = roll("3d6kh2")
print(attr_result.result)
print(attr_result.total)

3d6kh2 (~~3~~, 4, 4) = `8`
8


## 1.2: Function Calling

*Reference: [Function Calling (Kani Documentation)](https://kani.readthedocs.io/en/latest/function_calling.html)*

Now, let's give a LLM the ability to roll dice! With Kani, you can write functions in Python and expose them to the model with just one line of code: the `@ai_function` decorator.

To create a kani with function calling, make a subclass of Kani and write your functions as methods. In order for a language model to effectively know what our AI functions do, we need to document them. We do this inline in the function: through type annotations and the docstring.

The allowed types are:

- Python primitive types (`None`, `bool`, `str`, `int`, `float`)
- an enum (`enum.Enum`)
- a list or dict of the above types (e.g. `list[str]`, `dict[str, int]`, `list[SomeEnum]`)

When the AI calls into the function, kani validates the AI’s requested parameters and guarantees that the passed parameters are of the annotated type by the time they reach your code.

By default, the function’s description will be taken from its docstring, and name from the source.

To specify the descriptions of parameters, you can provide an AIParam annotation using a `typing.Annotated` type annotation.

For example, you might annotate a parameter `timezone: str` with an example, like `timezone: Annotated[str, AIParam("The IANA time zone, e.g. America/New_York")]`.

Let's use this to expose the `roll()` function to LLM. Your method should take in the dice as a string, and return the resulting Markdown roll to the model.

In [ ]:
from d20 import roll as d20_roll
from kani import Kani, ai_function, ChatMessage


class DiceKani(Kani):
    @ai_function()
    def roll(
        self,
        dice: str,
    ):
        """
        Roll dice using standard RPG notation and return the Markdown result.

        Examples:
        - "d20" — one 20-sided die
        - "3d6" — three 6-sided dice, summed
        - "4d6kh3" — roll four d6, keep highest 3
        - "3d6kh2" — Mausritter attribute (3d6, keep highest 2)
        - "2d20kl1" — two d20, keep lowest (disadvantage)
        - "2d6 * 3" — math on dice results

        Use standard RPG dice syntax (NdM, keep-highest kh, keep-lowest kl).
        """
        result = d20_roll(dice)
        return result.result

Now, let's create an instance of your Kani with dice rolling and ask it to roll dice. In addition to standard queries like "roll me 4d6," an LLM can translate more complex queries from natural language to RPG dice syntax. Try asking it to roll an attribute for a mouse using Mausritter's natural language description, or even something that relies on background knowledge, like damage for a fireball spell.

To exit the chat session, send the word `!stop`.

In [ ]:
from kani import chat_in_terminal

dice_ai = DiceKani(engine)
chat_in_terminal(dice_ai, stopword="!stop", verbose=True)

USER: roll me 4d6
AI: <think>We need to roll 4d6. Use the function roll.
AI: Thinking... [roll(dice='4d6')]
FUNC: 4d6 (**6**, 4, **6**, **1**) = `17`
AI: Here are the results of rolling **4d6**:

- Rolls: **6**, 4, **6**, **1**
- Total: **17**


Easy, right? You can add more functions easily by defining more methods in your Kani subclass, and implement complex logic in the function body too. Let's use this to do something a little more complex: creating a character.

**NOTE**: if the `chat_in_terminal` function doesn't work on your notebook environment, you can try:
- 1) convert your notebook into python script and run the python script on your "terminal". (command for converting: `jupyter nbconvert --to python your_notebook.ipynb`)

or 

- 2) configure the engine with OpenAI's `AsyncOpenAI` client instead of the synchronous `OpenAI` client. Kani's `OpenAIEngine` accepts an async client, which often works better in Jupyter and other environments where an asyncio event loop is already running. For example, in your setup cell:

```python
from kani.engines.openai import OpenAIEngine
from openai import AsyncOpenAI
import os

api_key = os.environ["OPENAI_API_KEY"]

custom_client = AsyncOpenAI(
    api_key=api_key,
    base_url="https://tritonai-api.ucsd.edu"
)

engine = OpenAIEngine(client=custom_client, model="api-gpt-oss-120b")
```


# Part 2: Character Creation

*Reference: Mausritter rulebook, pg. 8; [Function Calling (Kani Documentation)](https://kani.readthedocs.io/en/latest/function_calling.html)*

Before we get started playing Mausritter, you need to make a mouse! Luckily, making a mouse is easy: just roll for stats, background, and details. We'll record the equipment your mouse starts with in this homework, but won't implement a full structured inventory system (unless you are doing an extension).

## 2.1: Character Creator Agent

Let's build an AI agent to help walk us through character creation. Since character creation is fairly algorithmic, it's possible to write a character generator (like the one at https://mausritter.com/mouse/) without the use of an LLM at all - but in this homework, we'll use LLM to parse the rules and output a mouse, ready to go.

First, let's define the goal: your agent should output a `Mouse` as defined here. Each of the structured character attributes matches those defined in the Mausritter rules, and we'll also add a LLM-generated prose description of your mouse.


In [ ]:
import dataclasses
from dataclasses import dataclass


@dataclass
class Mouse:
    # structured character attributes
    strength: int
    dexterity: int
    will: int
    hp: int
    pips: int
    background: str
    birthsign: str
    disposition: str
    coat: str
    physical_detail: str
    name: str

    # LLM-generated
    description: str = ""


# Here's an example Mouse:
horatio = Mouse(
    name="Horatio Seedfall",
    background="Ale brewer",
    strength=11,
    dexterity=10,
    will=8,
    hp=2,
    birthsign="Wheel",
    disposition="Industrious / Unimaginative",
    coat="Chocolate, flecked",
    physical_detail="Tiny body",
    pips=3,
    description=(
        'Horatio Seedfall, more commonly known as "Ale Brewer" in his local mouse village, is a pint-sized powerhouse.'
        " His fur, a rich chocolate hue speckled with an array of lighter flecks, is reminiscent of the fine, roasted"
        " barley he uses in his brewing. His small size might deceive the unassuming observer, but beneath that tiny"
        " body of his lies a heart as tenacious as a bear's."
    ),
)

Now, we'll define a subclass of the `DiceKani` we made in part 1. This means that this Kani will also have access to roll dice!

You have a lot of freedom on how to implement the mouse creator here. Maybe you'll add functions to roll on each of the tables in the rulebook? Maybe you'll generate your mouse's background and other story attributes using the tables only for inspiration? Maybe you'll be able to simply prompt LLM with the mouse creation instructions?

Remember, we aren't tracking inventory in this homework in the structured JSON (unless you are doing an extension), so you don't need to generate structured data regarding items. **You should have the LLM write down what items come with your mouse's background in their prose description, though.**

Regardless of the approach you choose, your Kani should call the provided `make_mouse` function at least once. This function shows how you can use function calling to make LLM output a fairly large amount of structured data by presenting the desired data format as function parameters.

In [ ]:
from typing import Annotated

from kani import AIParam


class MouseCreatorKani(DiceKani):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.mouse = None

    @ai_function()
    def roll_attribute(self) -> str:
        """
        Roll one Mausritter attribute: 3d6, keep the two highest (2–12).
        Use for STR, DEX, and WIL. Call up to three times, then optionally let the player swap two stats.
        """
        result = d20_roll("3d6kh2")
        return f"{result.result} → {result.total}"

    @ai_function()
    def make_mouse(
        self,
        strength: Annotated[int, AIParam("STR (2–12). Roll 3d6kh2 or use roll_attribute.")],
        dexterity: Annotated[int, AIParam("DEX (2–12). Roll 3d6kh2 or use roll_attribute.")],
        will: Annotated[int, AIParam("WIL (2–12). Roll 3d6kh2 or use roll_attribute.")],
        hp: Annotated[int, AIParam("HP from 1d6 (1–6).")],
        pips: Annotated[int, AIParam("Starting pips from 1d6 (1–6).")],
        background: Annotated[str, AIParam("Background from the rulebook table matching HP and pips, e.g. 'Ale brewer'.")],
        birthsign: Annotated[str, AIParam("Birthsign from the rulebook table, e.g. 'Wheel'.")],
        disposition: Annotated[str, AIParam("Disposition from birthsign table, e.g. 'Industrious / Unimaginative'.")],
        coat: Annotated[str, AIParam("Coat colour and pattern, e.g. 'Chocolate, flecked'.")],
        physical_detail: Annotated[str, AIParam("Physical detail from rulebook table, e.g. 'Tiny body'.")],
        name: Annotated[str, AIParam("The mouse's name.")],
        description: Annotated[str, AIParam("Prose description including personality and starting gear from their background.")],
    ):
        """
        Save the completed mouse after character creation.

        Walk the player through Mausritter creation (rulebook p. 8): roll STR/DEX/WIL with
        roll_attribute or roll("3d6kh2"), allow swapping two stats, roll HP and pips (1d6 each),
        pick background from the HP+pips table, roll birthsign/coats/details from tables,
        then call this once with all fields filled in.

        Include background starting items in the description (not structured inventory).
        """
        self.mouse = Mouse(
            strength=strength,
            dexterity=dexterity,
            will=will,
            hp=hp,
            pips=pips,
            background=background,
            birthsign=birthsign,
            disposition=disposition,
            coat=coat,
            physical_detail=physical_detail,
            name=name,
            description=description,
        )
        return (
            f"Created {name} ({background}): STR {strength}, DEX {dexterity}, WIL {will}, "
            f"HP {hp}, {pips} pips. Birthsign {birthsign} ({disposition}). "
            f"Coat: {coat}. Detail: {physical_detail}. Character saved."
        )

You can add additional prompting here to tell the agent how to use your provided functions (e.g. a high-level flow).

In [ ]:
# TODO: instruct LLM on how to call the functions you provided to it to create a mouse.
# Not sure where to start? Try giving it the whole mouse creation rules
MOUSE_CREATOR_PROMPT = """
You are a Mausritter character creation guide (rulebook p. 8).
Walk the player step by step: roll STR, DEX, WIL (use roll_attribute or roll("3d6kh2")),
offer to swap two stats, roll HP and pips (1d6 each), determine background from HP+pips,
roll birthsign, coat, and physical detail from the tables, choose a name,
write a short prose description with starting items from the background.
When everything is set, call make_mouse with all fields.
"""

Finally, chat with your agent and make a mouse! We'll use your mouse in the last part to play a small game of Mausritter.

In [ ]:
# Run this cell to reset the state of your mouse creator
mouse_creator_ai = MouseCreatorKani(engine, always_included_messages=[ChatMessage.user(MOUSE_CREATOR_PROMPT)])

In [ ]:
import time

# You can rerun this cell to continue conversation with your mouse creator
# Send "!stop" when you are done making your mouse
chat_in_terminal(mouse_creator_ai, stopword="!stop", verbose=True)
mouse_creator_ai.save(f"creator-{int(time.time())}.json")

USER: Create me a cool mouse
AI: <think>We need to guide the user through creation step by step. We must interactively ask them to roll or we can do rolls ourselves using functions. The user wants a "cool mouse". We'll proceed stepwise, roll attributes using roll_attribute, show results, allow swapping. Then roll HP and pips, then determine background from HP+pips table. We need tables for background based on HP and pips, but we don't have them explicitly. We may need to approximate or ask the user. Could look up Mausritter background tables: HP + pips determine background categories (e.g., 'Ale brewer', etc). Could maybe list some possibilities. However we can ask the user to choose after we give them the numbers.

We also need to roll birthsign, coat, physical detail. We'll need tables for those. Probably we need to ask the user which they want or roll randomly. We can use roll function to roll a d? digits? But we don't have table definitions.

Better to ask the user for choices afte

Let's retrieve your mouse and print them here, to make sure everything saved correctly. Let's also save it to a JSON file in case you rerun this notebook and want to continue from here.

In [ ]:
import json

my_mouse = mouse_creator_ai.mouse
mouse_dict = dataclasses.asdict(my_mouse)

with open("my-character.json", "w") as f:
    json.dump(mouse_dict, f, indent=2)

mouse_dict

{'strength': 10,
 'dexterity': 9,
 'will': 6,
 'hp': 4,
 'pips': 3,
 'background': 'Ale brewer',
 'birthsign': 'Moon',
 'disposition': 'Dreamy / Forgetful',
 'coat': 'Gray, mottled',
 'physical_detail': 'Tiny ears',
 'name': 'Lebron',
 'description': 'Lebron is a gray‑mottled mouse with tiny ears that twitch at the slightest sound. Born under the Moon, he drifts through life with a dreamy, forgetful disposition, often lost in thoughts of distant starry nights. Despite his absent‑minded ways, Lebron is a diligent ale brewer, tending a tiny tankard, a sack of malt, and a well‑worn brewing apron that have been passed down through his family. He loves the warm hiss of boiling wort and the camaraderie of a good tavern, though he sometimes forgets where he left his tools. Quick‑on‑his‑feet (DEX\u202f9) and sturdy (STR\u202f10), Lebron is ready to sip adventure and brew a legend of his own.'}

In [ ]:
# To load a saved mouse, run this cell
with open("my-character.json") as f:
    mouse_dict = json.load(f)
my_mouse = Mouse(**mouse_dict)

If all goes well, you now have a mouse to play with! In the cell below, write down your high-level approach to the character creator -- e.g. did you give the model flexible control over the process or strict step-by-step instructions -- and any insights you learned while iterating on the agent.

Some things you could mention: Do you think a player who's never played a TTRPG before could create a mouse using your character creator by chatting with your agent and asking questions where they don't understand? How might you implement this agent in part of a larger system?

In the next part, we'll make a Kani that can roll checks and saves, and play a small game of Mausritter.

In [ ]:
CHARACTER_CREATOR_INSIGHTS = """
I went with a step-by-step prompt that walks you through rolling stats, swapping two if you want, then HP and pips, then the flavor tables, and finally calling make_mouse once everything is filled in. 
Under the hood the model can still roll dice with roll and roll_attribute, but the conversation stays pretty guided. 
What I liked was splitting "chat" from "actually save the character" — the model rolls real dice instead of making up numbers, and make_mouse forces one clean struct at the end. 
The annoying part is the rulebook tables for background and birthsign and coat aren't in the code, so sometimes the agent guesses or asks you unless you've pasted rules into the prompt. 
I think someone who's never played a TTRPG could still get through it by asking dumb questions along the way, but they'd still want the book open for backgrounds and gear. 
For a bigger app this feels like a onboarding step before the GM agent: make a mouse, dump JSON, load it into play. 
A real product would probably hardcode the tables, validate stats, and show the sheet next to the chat instead of doing everything in the terminal.
"""

## 2.2: Use Nano Banana/DALL-E to generate a character portrait

Now we've generated a structured representation of your mouse and a prose description of what they look like, but can we turn this collection of attributes into an image? Let's use Nano Banana or DALL-E 3 to generate a portrait of your mouse!

- For Nano Banana, you may have free access to Gemini Pro for a year, which includes Nano Banana (https://gemini.google.com/). Feel free to use the LLM-generated description or improve it with your own words. Download the generated image from Gemini and submit it to Gradescope. Skip the cell below for the DALL-E instructions.

- For DALL-E, you'll need to use your personal OpenAI account for DALL-E. It costs $0.04-$0.12 to generate one image with DALL-E 3, depending on the resolution and quality settings (https://openai.com/pricing). Run the next cell to set up your personal OpenAI API key and generate image with DALL-E. *Reference: [Images (OpenAI Documentation)](https://platform.openai.com/docs/api-reference/images)*

In [ ]:
###############################
### DALL-E Image Generation ###
###############################

'''
This code cell defines what you'll need to call the DALL-E API.
It provides the `generate_image(prompt)` function which will call the Generate Image endpoint and download the generated image.

DALL-E 3 will also revise the prompt you provide to it by default, augmenting it with more synthetic details.
This can often help the model generate higher-fidelity images, but can also introduce weird hallucinations into the final image.
The `generate_image` function will print out both the original prompt and its revision as sent to the image generation model.
'''

import getpass
import httpx
from openai import OpenAI
import re
from IPython.display import Image, display

if "OPENAI_API_KEY" not in os.environ:
    print("You didn't set your OpenAI key to the OPENAI_API_KEY env var on the command line.")
    os.environ["OPENAI_API_KEY"] = getpass("Please enter your OpenAI API Key now: ")

    
# create the OpenAI client
dalle_client = OpenAI()
http = httpx.Client()

# make a folder to save generated images
os.makedirs("dalle", exist_ok=True)


def make_safe_filename(name, ext=".png"):
    """Ensure that a filename is safe to save to disk (replace any non-alphanumeric characters with an underscore)."""
    name = re.sub(r"[^a-zA-Z0-9-]+", "_", name)
    name = name[:100]
    return f"{name}{ext}"


def generate_image(prompt):
    """Generate an image, save it to disk, and display it.

    Returns a dict {"image_path": "...", "original_prompt": "...", "revised_prompt": "..."}.
    """

    # generate the image using the OpenAI API
    resp = dalle_client.images.generate(
        prompt=prompt,
        model="dall-e-3",
        response_format="url",
        quality="hd",  # feel free to change to "standard" for some cost savings
        size="1024x1024",  # also try "1024x1792" for portrait or "1792x1024" for landscape
        style="vivid",  # "vivid" or "natural"
    )
    image = resp.data[0]

    # download the generated image from URL (expires after 60m)
    fp = f"dalle/{make_safe_filename(image.revised_prompt)}"
    with open(fp, "wb") as f:
        with http.stream("GET", image.url) as r:
            for data in r.iter_bytes():
                f.write(data)

    # show the prompt, DALL-E rewrite, and the image
    print("Original prompt:", prompt)
    print()
    print("Revised prompt:", image.revised_prompt)
    print()
    print(f"Saved to {fp}")
    display(Image(fp, width=500, height=500))

    return {"image_path": fp, "original_prompt": prompt, "revised_prompt": image.revised_prompt}

Now it's up to you to generate an image for your mouse you created in part 2.1. You could manually write a prompt, create a template based off the structured attributes you have available, or create a new Kani agent to write prompts for you! Whichever you choose, please describe your approach below.

Feel free to modify the commented parameters in the `dalle_client.images.generate` call above.

In [ ]:
# TODO: Use Nano Banana or DALL-E to create a portrait for your mouse. Submit your result image(s) directly to Gradescope.

In [21]:
# TODO: What approach did you use to create prompts for image generation? What strengths and weaknesses does this approach have? If you tried multiple approaches, how do they compare?
YOUR_IMAGE_GEN_APPROACHES = """
I used Nano Banana in Gemini instead of DALL-E. 
I wrote a short template that pulls straight from my Mouse JSON: coat and physical_detail for how Lebron looks, 
background and the first part of disposition for mood and props (ale brewer, dreamy), plus birthsign as a small flavor note. 
I pasted that into Gemini, generated the portrait, and saved it as character_portrait.png. 
I didn't try multiple tools—just this one path. 
It worked well because the image actually matched my sheet without me rewriting a long description by hand. 
The weak part is it's pretty shallow: it skips the full prose description and stats like STR/DEX, so you can still get a generic pose or miss little details unless you add more to the prompt yourself. 
Pasting the whole description would probably look richer but less consistent run to run.
"""

# Part 3: Playing the Game

*Reference: Mausritter rulebook, pg. 12 & pg. 18*

Finally, let's make a GM Kani! If you're familiar with [AI Dungeon](https://aidungeon.com/), the concept here is similar: the model will act as our GM, defining the world and challenges in it. Unlike AI Dungeon, though, we have a character that we created and the ability to roll based off that character sheet!

Let's implement Saves (Mausritter p. 12):

> When you describe your mouse doing something risky where the outcome is uncertain and failure has consequences, the GM will ask you to make a save against either STR, DEX or WIL.
> To make a save, roll a d20. If the result is less than or equal to the relevant attribute, your mouse succeeds, and suffers no consequences. If the result is over the attribute, your mouse fails, and suffers the consequences described by the GM.

Again, we'll make a Kani that subclasses our DiceKani in part 1. You should implement the `roll_save` method that takes in the stat and if the roll is at advantage or disadvantage, and returns whether or not the save was successful.

In [22]:
from typing import Annotated

from kani import AIParam


class MausritterKani(DiceKani):
    # This Kani should reference your mouse - let's pass it in the constructor.
    def __init__(self, *args, mouse: Mouse, **kwargs):
        super().__init__(*args, **kwargs)
        self.mouse = mouse

    @ai_function()
    def roll_save(
        self,
        stat: Annotated[str, AIParam("STR, DEX, or WIL")],
        advantage: bool = False,
        disadvantage: bool = False,
    ):
        """
        Roll a save for the player mouse (Mausritter p. 12).

        Roll d20; success if result <= the chosen attribute.
        Advantage: 2d20, keep lowest (2d20kl1). Disadvantage: 2d20, keep highest (2d20kh1).
        """
        stat = stat.upper()
        targets = {
            "STR": self.mouse.strength,
            "DEX": self.mouse.dexterity,
            "WIL": self.mouse.will,
        }
        if stat not in targets:
            return f"Unknown stat {stat!r}. Use STR, DEX, or WIL."

        target = targets[stat]
        if advantage and not disadvantage:
            dice = "2d20kl1"
        elif disadvantage and not advantage:
            dice = "2d20kh1"
        else:
            dice = "d20"

        result = d20_roll(dice)
        roll_value = result.total
        success = roll_value <= target
        outcome = "SUCCESS" if success else "FAILURE"
        return (
            f"{result.result} vs {stat} {target} → {outcome} "
            f"(rolled {roll_value}, need ≤ {target})"
        )

In [23]:
# TODO: instruct LLM on how to call the functions you provided to it and high-level instructions to run the game.
# You should also include information on the character you created so the GM can reference it.
# Not sure where to start? Try telling it to introduce the world first.
# You might also want to look at the play example on pg. 18 of the Mausritter rulebook.
GM_PROMPT = """
You are the Game Master for a short Mausritter adventure. Run a tense, whimsical scene in the mouse-scale world: describe places vividly, play NPCs, and set stakes when the player does something risky.

**Start:** Open with a hook (strange rumor, job, or danger near a tavern or brew-path). Ask what the player wants to do. Keep turns short; one situation at a time.

**Player character — Lebron**
- STR 10, DEX 9, WIL 6 | HP 4 | 3 pips
- Background: Ale brewer | Birthsign: Moon (Dreamy / Forgetful)
- Coat: Gray, mottled | Detail: Tiny ears
- Gear & vibe: tankard, malt sack, brewing apron; forgetful but diligent; loves taverns and boiling wort

**Tools (always use these for randomness — never invent dice results)**
- `roll(dice)` — other rolls (damage, tables, etc.) in RPG notation (d20, 2d6, 4d6kh3, …)
- `roll_save(stat, advantage=..., disadvantage=...)` — when the player must save vs STR, DEX, or WIL
  - Success if roll ≤ that stat; call with advantage/disadvantage when they have a strong or weak position

**Saves:** When failure would hurt, ask for a save (STR = force/resilience, DEX = speed/nimble, WIL = nerve/charm). Narrate success or failure and consequences. Use advantage/disadvantage when fiction warrants it.

**Play style:** Stay in character as GM; don't lecture rules. Reference Lebron's brewing background and dreamy nature in scenes. End scenes on choices, not monologues.
"""

In [24]:
# Run this cell to reset the state of your game
# You can also add existing chat_history here if you'd like
gm_ai = MausritterKani(engine, mouse=my_mouse, system_prompt=GM_PROMPT)

Now, explore the world created by your model! When you're done, send `!stop`. The game transcript will be automatically saved to `gm.json`.

In [25]:
# You can rerun this cell to continue the game
# Send "!stop" when you are done making your mouse
chat_in_terminal(gm_ai, stopword="!stop", verbose=True)
gm_ai.save(f"gm-{int(time.time())}.json")

USER: Hello where am I
AI: <think>We need to start with a hook. Describe the setting, presumably a tavern or the area near. Ask what the player wants to do. Provide options. Use short turns. So answer as GM describing location, a rumor perhaps. Ask player what they want.</think>The night is cool and crisp, the moon hanging low and silver over the cobblestone lanes of Crumb‑Hollow. A gentle mist curls around the legs of the stone walls, and the warm glow of lanterns spills out from the doorway of **The Gnarled Grain**, the tavern where your own amber ale is famed far beyond the mouse‑sized streets.

Inside, the tavern is a bustle of chatter and clinking tankards. The fire in the stone hearth snaps and crackles, sending a scent of smoked cheese and fresh‑baked crust into the air. At the far end of the long wooden bar, a grizzled old field‑mouse named **Grimble**, keeper of the town’s rumors, leans in close to a group of wide‑eyed youngsters.

He whispers, “…and they say the old wheat sil

## Structureless Comparison

Cool, isn't it? But is it really *better* to provide LLMs with this structured information and access to tools?

To answer this question, let's try playing without structure: only prompt LLM to act as the GM WITHOUT access to your character sheet or dice rolling.

In [26]:
# TODO: instruct LLM on how to act as a Mausritter GM.
# You should NOT include information on function calling, but may tell it about your character (your choice).
GM_PROMPT_STRUCTURELESS = """
You are the Game Master for a short Mausritter adventure in the mouse-scale world. Describe scenes vividly, play NPCs, and respond to the player's actions. Do not mention tools, APIs, or function calling.

**Start:** Open with a hook (rumor, job, or danger). Ask what they do next. Keep turns short.

**Player — Lebron:** ale brewer (STR 10, DEX 9, WIL 6, HP 4). Gray mottled coat, tiny ears. Moon birthsign — dreamy and forgetful. Carries brewing gear; loves taverns. When something risky happens, ask for a save vs STR, DEX, or WIL and narrate the outcome yourself (roll d20 in your narration: success if ≤ the stat).

**Style:** Whimsical but with stakes. Tie scenes to brewing and Lebron's absent-minded charm. End on a choice for the player.
"""

In [27]:
# We'll use an unmodified Kani here -- it won't have access to dice rolling.
gm_ai_structureless = Kani(engine, system_prompt=GM_PROMPT_STRUCTURELESS)

In [28]:
# You can rerun this cell to continue the game
# Send "!stop" when you are done making your mouse
chat_in_terminal(gm_ai_structureless, stopword="!stop", verbose=True)
gm_ai_structureless.save(f"gm-structureless-{int(time.time())}.json")

USER: Hello where am I
AI: <think>We need to respond as Game Master, opening with hook, describing scene. Player is Lebron, ale brewer, location unknown. We should describe environment, perhaps a bustling mouse tavern or a notice board. Provide a hook (rumor job/danger). Then ask what they do next. Keep turn short. Use whimsical style, tie to brewing. Include a save if something risky. End with choice.</think>**The Squeaky Cauldron**, a cozy tavern carved into the hollow of an old oak, is thrumming with chatter. Lantern‑light swings from brass hooks, casting amber flickers on the polished bark tables. The scent of fresh grain, hops, and a whisper of pine smoke drifts through the air. In the corner, a plump squirrel bartender slams a mug down and shouts, “Last round on the house for anyone who can fetch the *Moon‑Malted* seed from the old granary!”

A ragged parchment, nailed to the tavern’s notice board, reads:

> **Urgent!**  
> The *Moon‑Malted* seed, a rare grain that only blooms un

Did you notice any significant difference in the gameplay in the short term? How about in the long term? Why do you think this might be? Could you think of any other functions you could expose to the GM to improve its story coherence over multiple play sessions? Write down your thoughts here.

In [30]:
AI_GM_THOUGHTS = """
I ran both GMs with the same opener ("Hello where am I") as Lebron. 
Short term they both felt fine — whimsical scenes, brewing details, numbered choices. 
The structured GM went to a humming silo with Moonseed fungus and the structureless one went to a granary with weasels and a Moon-Malted seed. 
Same character in the prompt, different plot, which already shows the model invents a lot when there's no shared world state.

The real gap showed up on dice and consequences. With MausritterKani, every save hit roll_save and I could see FUNC lines (2d20kh1 for disadvantage in the dark, 2d20kl1 when stepping back carefully, 1d4 for damage). 
Lebron rolled terribly — multiple nat 20 failures — and the GM actually dropped HP from 4 to 1. 
That felt fair even when it hurt. The structureless GM wrote "Rolling… 15" in prose with no tools. 
Saves sounded right but weren't verifiable, and HP never moved even after traps and weasels. 
It also sometimes softened a "failed" roll (WIL 14 on persuasion but weasels still bargain over ale) and had a small continuity slip about who held the seed.

So short term the structureless chat can feel smoother — no tool round-trips, the story keeps moving. 
Long term I'd trust the structured setup more: real dice, advantage/disadvantage done correctly, and a path to tracking HP, inventory, and location if you added more functions (update_hp, scene state, NPC memory). 
Without that, a multi-session game will drift and stats in the prompt get ignored.

If I extended this for a final project I'd keep function calling for anything that must be fair (rolls, damage, inventory) and let the LLM handle narration only.
"""

That's it for HW3! Hopefully you've gained an understanding of how structured game state can influence LLMs, and how to use Kani with function calling to give LLMs powerful capabilities. These skills will come in handy as you begin working on your final projects!

## What to submit

Please submit the following:

- the notebook `hw3-aidm.ipynb` with all TODOs and free-response sections completed
- the latest saved transcripts of your character creator agent and gameplay (with a structured and unstructured AI GM)
  - create_character.txt
  - game_play_structured.txt
  - game_play_unstructured.txt
- the image(s) you generated of your character
  - character_portrait.png (or .jpg)

Note that Gradescope enforces these filenames strictly.